# Lexsi Track 3 — fresh run (fp16 fix)
Clone goes into `lexsi-v2`, so the old broken copy is never used.
Run cells top to bottom. Stop after cell 9 first; training comes later.


In [ ]:
!nvidia-smi

In [ ]:
%cd /content
![ -d lexsi-v2 ] && (cd lexsi-v2 && git pull) || git clone https://github.com/PRINCE2-AI/lexsi-agentic-hinglish.git lexsi-v2
%cd lexsi-v2

In [ ]:
# 3-6 min. Dependency-conflict warnings come from Colab's preinstalled stack.
!pip install -q -r requirements.txt

**Restart the runtime now** (Runtime > Restart session) so torch loads cleanly.
After restart, continue from the next cell — do not run the install cell again.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from pathlib import Path

script = Path("scripts/train_grpo.py").read_text()
assert "model_init_kwargs" in script, "OLD CODE — run the clone cell again"
print("fp16 fix present")
!git log --oneline -1

In [ ]:
# No model needed. Expect: 20/20 success, all scaffold tests passed.
!python data/build_tasks.py
!python scripts/run_oracle_eval.py
!python tests/test_scaffold.py

## Baseline (the left column)
About 5-10 minutes. Keep the summary.json output.

In [ ]:
!python scripts/run_baseline.py --model CohereLabs/tiny-aya-global --limit 20

## GRPO smoke test
If the previous OOM left memory dirty, restart the runtime, re-run the login cell, then this cell.

In [ ]:
!python scripts/train_grpo.py --max-steps 2

In [ ]:
# Only after 2 steps pass without OOM.
!python scripts/train_grpo.py --max-steps 10

In [ ]:
# Tight-memory fallback if the T4 still OOMs:
# !python scripts/train_grpo.py --max-steps 10 --max-completion-length 128 --max-steps-per-turn 2

## Real run — use Kaggle 2xT4, not free Colab.
Leave this commented here.

In [ ]:
# !python scripts/train_grpo.py --max-steps 120 --efficiency

## Tuned evaluation (after a real run)

In [ ]:
# !python scripts/run_baseline.py --model CohereLabs/tiny-aya-global #     --adapter runs/grpo_tiny_aya --out runs/tuned

## After the runs
Copy `runs/baseline/summary.json` and `runs/tuned/summary.json` into the table in README.md and docs/hf_article.md.
Upload the adapter, tokenizer, dataset and trajectories.jsonl to Hugging Face.
Deploy `space/app.py` as a GPU Space with `ADAPTER_ID` set.
Submit before Nov 14 end of day.